# NB18 — Canonical EfficientNet coupling CI (reproduce-then-extend)

**Goal.** Produce a trustworthy per-generator bootstrap CI for the EfficientNet competence–calibration coupling, so the manuscript can drop the "per-generator calibration values not retained" note and report an interval for all three architectures.

**Why this notebook is careful.** The committed Xception coupling is r = −0.88, computed from `reports/calibration/unified_trust_signals.csv`. Independent attempts to recompute that ECE_cal from the raw score parquets give r ≈ −0.64 to −0.70 — i.e., a generic oracle/in-sample calibration does **not** reproduce the canonical numbers. The calibration step that made the committed file uses a specific protocol. Therefore this notebook does not invent a calibration method. It:

1. **Locates the original code** that produced `unified_trust_signals.csv` (the notebook with the calibration cell), and reuses that exact `compute_calibrated_ece` function.
2. **Verifies** the reused function reproduces the committed Xception per-generator ECE_cal (mean abs error < 0.005 and r within 0.02 of −0.88) **before trusting it**.
3. Only if verification passes, **applies the identical function to EfficientNet's 20 DF40 generators**, writes the canonical `unified_trust_signals_effnet.csv`, and bootstraps the coupling CI.

If step 2 fails, the notebook stops and tells you which calibration cell to point it at. It will never report an EfficientNet CI computed by a method that can't reproduce Xception.

Run Cell 0 → 1 → 2 → 3 → 4 → 5. If Cell 2 fails its assertion, read its message and set `CALIB_SOURCE` in Cell 2 to the right notebook before continuing.

In [8]:
# CELL 0 — setup: mount Drive, restore git identity, config
import os, shutil, subprocess, glob, json
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

CDTS_ROOT = '/content/drive/MyDrive/CDTS_Research'
REPO      = f'{CDTS_ROOT}/deepfake-trust-research'
for src, dst in [(f'{CDTS_ROOT}/.git-credentials', '/root/.git-credentials'),
                 (f'{CDTS_ROOT}/.gitconfig',       '/root/.gitconfig')]:
    if os.path.exists(src):
        shutil.copy(src, dst)
print('git identity restored' if os.path.exists('/root/.gitconfig') else 'WARNING: no .gitconfig')

CFG = dict(
    repo     = REPO,
    reports  = f'{REPO}/reports/calibration',
    scores   = f'{REPO}/reports/scores',
    notebooks= f'{REPO}/notebooks',
    src      = f'{REPO}/src',
    n_boot   = 5000,
    seed     = 42,
)
os.chdir(CFG['repo'])
print('cwd:', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
git identity restored
cwd: /content/drive/MyDrive/CDTS_Research/deepfake-trust-research


## Cell 1 — find the code that produced `unified_trust_signals.csv`

Searches the repo for the notebook/script that writes `unified_trust_signals.csv` and for a reusable calibration function (e.g. `calibrated_ece`, `oracle_ece`, `compute_ece`, `fit_calibrator`). Prints candidates so Cell 2 can reuse the real one rather than guessing.

In [9]:
# CELL 1 — locate the canonical calibration code in the repo
import re

def grep_repo(pattern, exts=('.py', '.ipynb')):
    hits = []
    for root, _, files in os.walk(CFG['repo']):
        if '/.git' in root: continue
        for fn in files:
            if fn.endswith(exts):
                p = os.path.join(root, fn)
                try:
                    txt = open(p, encoding='utf-8', errors='ignore').read()
                except Exception:
                    continue
                if re.search(pattern, txt):
                    hits.append(p)
    return hits

print("Files that WRITE unified_trust_signals.csv:")
writers = grep_repo(r"unified_trust_signals(?!_clip|_effnet)\.csv")
for w in writers: print("  ", w.replace(CFG['repo'], '.'))

print("\nFiles defining a calibration function:")
calib_files = grep_repo(r"def\s+\w*(calib|ece|platt|isotonic)\w*\s*\(")
for c in calib_files: print("  ", c.replace(CFG['repo'], '.'))

print("\nFiles defining ECE computation:")
ece_files = grep_repo(r"def\s+\w*ece\w*\s*\(")
for c in ece_files: print("  ", c.replace(CFG['repo'], '.'))

print("\n-> Note the file that BOTH writes unified_trust_signals.csv AND defines")
print("   the calibration. That is the canonical source for Cell 2.")

Files that WRITE unified_trust_signals.csv:
   ./notebooks/NB10b_explanation_rankcorr.ipynb
   ./notebooks/NB12_paper_figures.ipynb
   ./notebooks/NB13_effnet_robustness.ipynb
   ./notebooks/NB14_third_arch_signals.ipynb
   ./notebooks/NB17_coupling_CIs_and_effnet_dispersion.ipynb
   ./notebooks/NB18_canonical_effnet_coupling_CI.ipynb

Files defining a calibration function:
   ./external/DeepfakeBench/training/dataset/utils/warp.py
   ./notebooks/GBDF_download.ipynb
   ./notebooks/NB07_calibration_equity.ipynb
   ./notebooks/NB12_paper_figures.ipynb
   ./notebooks/NB13_effnet_robustness.ipynb
   ./notebooks/NB14_third_arch_signals.ipynb
   ./notebooks/NB15_dfd_coupling.ipynb
   ./notebooks/NB17_coupling_CIs_and_effnet_dispersion.ipynb
   ./notebooks/NB18_canonical_effnet_coupling_CI.ipynb
   ./src/metrics.py
   ./src/calibration.py
   ./src/calibrate_scores.py

Files defining ECE computation:
   ./external/DeepfakeBench/training/dataset/utils/warp.py
   ./notebooks/NB07_calibration_equ

## Cell 2 — reuse NB14's exact calibration path and VERIFY on Xception

This imports your repo's `metrics` and `calibration` modules and reproduces the **exact** per-generator computation from `NB14_third_arch_signals.ipynb`:

```
ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=0.5, seed=42)
pcal, _   = cal.fit_predict('hybrid', p[ci], y[ci], p[ti], switch_threshold_n=1000)
auc       = metc.roc_auc(p[ti], y[ti])        # AUC on the TEST split
ece_cal   = metc.ece(pcal, y[ti], 15, 'equal_mass')
```

It then asserts this reproduces the committed Xception `unified_trust_signals.csv` (mean |ΔECE| < 0.01, coupling r within 0.03 of −0.88) **before** applying it to EfficientNet. No manual configuration needed — the path is now known.

In [10]:
# ============================================================================
# CELL 2 (canonical) — reuse the EXACT calibration path from NB14 and verify
# ============================================================================
import sys, importlib, pandas as pd, numpy as np
from scipy.stats import pearsonr

# import the repo's real modules the same way NB14 does
for p in (CFG['src'], CFG['repo']):
    if p not in sys.path:
        sys.path.insert(0, p)
for k in list(sys.modules.keys()):
    if k in ("metrics", "calibration", "data_prep") or k.startswith("metrics."):
        del sys.modules[k]
import metrics as metc
import calibration as cal
print("imported repo metrics + calibration modules")

def canonical_auc_ece(d, calib_frac=0.5, seed=42):
    """EXACT NB14 per-generator computation:
       leakage_safe_split -> hybrid calibrator (Platt/iso switch at n=1000)
       -> AUC and ECE both on the TEST split (equal-mass, 15 bins)."""
    y = d['label'].to_numpy()
    p = d['prob_fake'].to_numpy()
    g = d['identity_id'].astype(str).to_numpy() if 'identity_id' in d.columns else None
    ci, ti, _ = cal.leakage_safe_split(y, groups=g, calib_frac=calib_frac, seed=seed)
    pcal, _ = cal.fit_predict("hybrid", p[ci], y[ci], p[ti], switch_threshold_n=1000)
    auc = metc.roc_auc(p[ti], y[ti])
    ece_cal = metc.ece(pcal, y[ti], 15, 'equal_mass')
    return auc, ece_cal

# ---- VERIFY against the committed Xception unified file before trusting it ----
U = pd.read_csv(f"{CFG['reports']}/unified_trust_signals.csv")
U['m'] = U['method'].str.lower()
committed_ece = dict(zip(U['m'], U['ECE_cal']))
committed_auc = dict(zip(U['m'], U['AUC']))

import glob, os
xfiles = sorted(glob.glob(f"{CFG['scores']}/xceptionFS_df40_*.parquet"))
rows = []
for f in xfiles:
    gen = os.path.basename(f).replace('xceptionFS_df40_', '').replace('.parquet', '').lower()
    if gen not in committed_ece:
        continue
    d = pd.read_parquet(f)
    if d['label'].nunique() < 2:
        continue
    auc, ece = canonical_auc_ece(d)
    rows.append(dict(method=gen, AUC=round(auc,4), ECE_recomp=round(ece,4),
                     AUC_committed=committed_auc[gen], ECE_committed=committed_ece[gen]))
chk = pd.DataFrame(rows)
chk['dECE'] = (chk['ECE_recomp'] - chk['ECE_committed']).abs()
chk['dAUC'] = (chk['AUC'] - chk['AUC_committed']).abs()
print(chk.to_string(index=False))

mad_ece = chk['dECE'].mean()
mad_auc = chk['dAUC'].mean()
r_recomp = pearsonr(chk['AUC'], chk['ECE_recomp'])[0]
r_comm   = pearsonr(chk['AUC_committed'], chk['ECE_committed'])[0]
print(f"\nmean|ΔECE| = {mad_ece:.4f}   mean|ΔAUC| = {mad_auc:.4f}")
print(f"r recomputed = {r_recomp:.3f}   committed = {r_comm:.3f}")

assert mad_ece < 0.01, f"ECE mismatch {mad_ece:.4f} >= 0.01 — calibration path differs from NB14."
assert abs(r_recomp - r_comm) < 0.03, f"coupling r mismatch ({r_recomp:.3f} vs {r_comm:.3f})."
print("\nVERIFIED: this path reproduces the committed Xception coupling. Safe to apply to EfficientNet.")


imported repo metrics + calibration modules
     method    AUC  ECE_recomp  AUC_committed  ECE_committed  dECE  dAUC
        dit 0.5214      0.1608         0.5214         0.1608   0.0   0.0
        sit 0.5535      0.0966         0.5535         0.0966   0.0   0.0
  stylegan2 0.6491      0.0629         0.6491         0.0629   0.0   0.0
  stylegan3 0.6792      0.0596         0.6792         0.0596   0.0   0.0
 styleganxl 0.6374      0.0932         0.6374         0.0932   0.0   0.0
  blendface 0.9429      0.0225         0.9429         0.0225   0.0   0.0
       ddim 0.7661      0.0411         0.7661         0.0411   0.0   0.0
 facedancer 0.9327      0.0254         0.9327         0.0254   0.0   0.0
   faceswap 0.8919      0.0455         0.8919         0.0455   0.0   0.0
facevid2vid 0.8216      0.0637         0.8216         0.0637   0.0   0.0
       fomm 0.8008      0.0544         0.8008         0.0544   0.0   0.0
      fsgan 0.9004      0.0364         0.9004         0.0364   0.0   0.0
     in

## Cell 3 — apply the verified calibration to EfficientNet (20 generators)

Uses the *same* verified `calibrated_ece` on EfficientNet's 20 DF40 score parquets, builds the canonical `unified_trust_signals_effnet.csv` (AUC, ECE_cal, entropy, dispersion per generator), and reports the coupling.

In [11]:
# ============================================================================
# CELL 3 (canonical) — apply the VERIFIED path to EfficientNet's 20 generators
# ============================================================================
eff_files = sorted(glob.glob(f"{CFG['scores']}/effnetb4_df40_*.parquet"))
print(f"EfficientNet DF40 parquets: {len(eff_files)}")
assert len(eff_files) >= 12, "fewer than 12 EffNet parquets; score the full DF40 set first."

# reference-free signals exactly as NB14 defines them (entropy, dispersion)
def reference_free_signals(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    entropy = float((-(p*np.log(p) + (1-p)*np.log(1-p))).mean())
    dispersion = float(p.std())
    return {'entropy': entropy, 'dispersion': dispersion}

rows = []
for f in eff_files:
    gen = os.path.basename(f).replace('effnetb4_df40_', '').replace('.parquet', '')
    d = pd.read_parquet(f)
    if d['label'].nunique() < 2:
        continue
    auc, ece = canonical_auc_ece(d)                 # SAME verified function as Xception
    rf = reference_free_signals(d['prob_fake'].to_numpy())
    rows.append(dict(method=gen, AUC=round(auc,4), ECE_cal=round(ece,4),
                     entropy=round(rf['entropy'],4), dispersion=round(rf['dispersion'],4), n=len(d)))

E = pd.DataFrame(rows).sort_values('AUC').reset_index(drop=True)
print(E.to_string(index=False))
out = f"{CFG['reports']}/unified_trust_signals_effnet.csv"
E.to_csv(out, index=False)
print(f"\nwrote canonical {out.replace(CFG['repo'],'.')}")

r_eff = pearsonr(E['AUC'], E['ECE_cal'])[0]
print(f"\nEfficientNet coupling r(AUC, ECE_cal) = {r_eff:.3f} over {len(E)} generators  (headline -0.83)")


EfficientNet DF40 parquets: 20
     method    AUC  ECE_cal  entropy  dispersion     n
  StyleGAN2 0.4658   0.1808   0.3326      0.3229 33794
     pixart 0.4721   0.2054   0.2907      0.3631 33794
  StyleGAN3 0.4765   0.2063   0.3355      0.3286 33794
 StyleGANXL 0.4768   0.1910   0.3703      0.2943 33794
  sadtalker 0.4894   0.2538   0.3599      0.2991 26606
        DiT 0.5073   0.1558   0.3794      0.3093 33794
        SiT 0.5346   0.1241   0.3893      0.3157 33794
    wav2lip 0.5501   0.2545   0.3481      0.3364 26434
        lia 0.6509   0.1726   0.3927      0.3350 25426
   pirender 0.6648   0.1808   0.4435      0.3064 25805
facevid2vid 0.7113   0.1983   0.4473      0.3124 25510
     inswap 0.7214   0.0992   0.3764      0.3581 19053
       ddim 0.7450   0.0318   0.3791      0.3498 33794
       fomm 0.7668   0.1154   0.4617      0.3110 25898
      sd2.1 0.7767   0.0334   0.3387      0.3570 33794
 facedancer 0.8588   0.0700   0.3131      0.3672 25872
    simswap 0.8607   0.0449   0.30

## Cell 4 — bootstrap the EfficientNet coupling CI

Percentile bootstrap (5,000 resamples) over the 20 per-generator points, matching the Xception/CLIP interval method. Also refreshes the full three-architecture CI table.

In [12]:
# ============================================================================
# CELL 4 (canonical) — bootstrap the EffNet CI + refresh the three-arch table
# ============================================================================
def boot_ci_r(x, y, n_boot=CFG['n_boot'], seed=CFG['seed']):
    x = np.asarray(x, float); y = np.asarray(y, float)
    rng = np.random.RandomState(seed); rs = []; n = len(x)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if np.std(x[idx]) > 0 and np.std(y[idx]) > 0:
            rs.append(pearsonr(x[idx], y[idx])[0])
    return float(np.mean(rs)), float(np.percentile(rs, 2.5)), float(np.percentile(rs, 97.5))

def coupling_ci_from(fname):
    d = pd.read_csv(f"{CFG['reports']}/{fname}").dropna(subset=['AUC','ECE_cal'])
    (r, lo, hi) = boot_ci_r(d['AUC'], d['ECE_cal'])
    return r, lo, hi, len(d)

table = []
for label, fname in [('Xception', 'unified_trust_signals.csv'),
                     ('EfficientNet-B4', 'unified_trust_signals_effnet.csv'),
                     ('CLIP-ViT', 'unified_trust_signals_clip.csv')]:
    r, lo, hi, n = coupling_ci_from(fname)
    table.append(dict(architecture=label, r=round(r,3), ci_lo=round(lo,2), ci_hi=round(hi,2), n=n))
T = pd.DataFrame(table)
print("\nThree-architecture coupling with bootstrap CIs:")
print(T.to_string(index=False))
T.to_csv(f"{CFG['reports']}/coupling_bootstrap_CIs.csv", index=False)
print("\nwrote coupling_bootstrap_CIs.csv")
print("\n>>> Paste this table back. If EffNet r is within ~0.02 of -0.83 with a tight CI,")
print("    the manuscript EffNet interval gets stated exactly like Xception/CLIP. <<<")



Three-architecture coupling with bootstrap CIs:
   architecture      r  ci_lo  ci_hi  n
       Xception -0.878  -0.95  -0.79 20
EfficientNet-B4 -0.833  -0.92  -0.70 20
       CLIP-ViT -0.860  -0.93  -0.75 20

wrote coupling_bootstrap_CIs.csv

>>> Paste this table back. If EffNet r is within ~0.02 of -0.83 with a tight CI,
    the manuscript EffNet interval gets stated exactly like Xception/CLIP. <<<


In [ ]:
# CELL 5 — commit + push the canonical artifacts
subprocess.run(['git', 'add',
                f"{CFG['reports']}/unified_trust_signals_effnet.csv",
                f"{CFG['reports']}/coupling_bootstrap_CIs.csv"], cwd=CFG['repo'])
subprocess.run(['git', 'commit', '-m',
                'NB18: canonical EfficientNet unified signals + per-arch coupling CIs'],
               cwd=CFG['repo'])
subprocess.run(['git', 'push'], cwd=CFG['repo'])
print('committed + pushed')